
### EX: Rapid Model Adaptation

In this exercise, we will load a pre-trained Keras image classification model (MobileNetV2), freeze its feature-extraction layers, and build a custom classification head for military cargo.

* **Load Base Model:** Import MobileNetV2 without its top classification layer.

* **Freeze Layers:** Set `trainable = False` for the base model.

* **Attach Custom Head:** Add a `GlobalAveragePooling2D` layer to flatten the data, and a new `Dense` output layer.

* **Inspect:** Print the model summary to prove that millions of parameters are frozen, and we only need to train a few thousand.

In [ ]:
#Only run this cell after selecting your kernel
!python.exe -m pip install --upgrade pip
!pip install numpy matplotlib scikit-learn pandas tensorflow

In [1]:

import numpy as np
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

# 1. Load the Pre-Trained Base Model (Weights learned from millions of images)
# include_top=False means we chop off the original 1000-class output layer
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Freeze the Base Model (Lock the weights)
base_model.trainable = False

# 3. Build the Custom Tactical Architecture
model = Sequential([
    base_model,                               # The frozen feature extractor
    GlobalAveragePooling2D(),                 # Flattens the 2D feature maps into a 1D vector
    Dense(64, activation='relu'),             # A new, trainable hidden layer
    Dense(1, activation='sigmoid')            # Custom output: 0 (Civilian Cargo), 1 (Military Transport)
])

# 4. Compile the Model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 5. Inspect the Architecture and Parameter Counts
print(model.summary())

# Notice in the output:
# Total params: ~2.3 million
# Trainable params: ~82,000 (Only the new Dense layers!)
# Non-trainable params: ~2.2 million (The frozen MobileNetV2 base)

# (In a real scenario, you would now call model.fit() using a folder of your specific drone images)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 82,049 (320.50 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

None



### Interpreting the Results

Look at the parameter count printed by `model.summary()`. The base model contains 2.2 million weights that already know how to identify edges, shadows, and vehicle chassis. Because they are marked as non-trainable, our computer's RAM and processors do not have to calculate gradients for them during backpropagation. We only have to train the 82,000 weights in our new custom head. This reduces the training time from weeks to minutes, enabling rapid, in-theater AI deployment.
